<h3>get raw image paths and labels</h3>

In [4]:
from src.data_loader import get_raw_image_paths_and_labels

image_paths, labels = get_raw_image_paths_and_labels()

<h3>get breif data statics</h3>

In [25]:
print(f"total images number : {len(image_paths)}\n")
print("images count per label: ")
for label in set(labels):
    print(f"|_{label}: {labels.count(label)}")

import cv2
unique_shapes = set()
for p in image_paths:
    img = cv2.imread(p)
    unique_shapes.add(img.shape)

print(f"\nTotal unique shapes: {len(unique_shapes)}")

total images number : 7023

images count per label: 
|_notumor: 2000
|_glioma: 1621
|_meningioma: 1645
|_pituitary: 1757

Total unique shapes: 387


<H3>processing images</h3>

In [30]:
import numpy as np
def preprocess_image(path, target_size=(224, 224), to_rgb=True):
    img = cv2.imread(path)
    if img is None:
        return None

    # OpenCV loads BGR by default; models expect RGB
    if to_rgb:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Adaptive interpolation based on resolution
    h, w = img.shape[:2]
    interp = cv2.INTER_AREA if (h >= target_size[1] and w >= target_size[0]) else cv2.INTER_LINEAR
    resized = cv2.resize(img, target_size, interpolation=interp)

    # Scale to [0.0, 1.0]
    normalized = resized.astype(np.float32) / 255.0
    return normalized

normalized_images = [preprocess_image(p) for p in image_paths]
X = np.stack(normalized_images, axis=0)

print(f"pictures shape: {X.shape}")

pictures shape: (7023, 224, 224, 3)


<h3>Encoding Labels</h3>

In [36]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

print("y type:", type(y))
print("y shape :", y.shape)

y type: <class 'numpy.ndarray'>
y shape : (7023,)


<h3>Split the Data</h3>

In [38]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

<h3>save the data into one comprissed file</h3>

In [41]:
from pathlib import Path
ROOT_DIR = Path.cwd().parent
output_dir = ROOT_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    output_dir / "dataset_splits.npz",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)